# ASVspoof 5 Track 1 - 데이터 분석

`data/dataset.py`의 로딩/전처리가 실제 데이터에서 어떻게 동작하는지 확인한다. 경로 규약은 `ASVspoof5_colab_subset.ipynb`와 같다 (Colab + Drive).

`ASVspoof5Track1Dataset` 전처리 요약
- `_load_audio`: mono 변환 후 `sr != target_sr`이면 `np.interp` 선형보간으로 리샘플
- `_crop_or_pad` (기준 길이 `n = round(sample_rate * clip_seconds)`):
  - `len >= n`: crop. train은 랜덤 시작점, eval은 중앙 crop
  - `len < n`: pad. `len >= 2`면 reflect, `len < 2`면 0으로 constant pad

확인할 것
1. 샘플레이트 분포 (리샘플이 실제로 일어나는지)
2. 길이 분포와 기준 길이 대비 미달/초과 개수
3. 미달 발화 중 reflect / constant 패딩 개수
4. 샘플 하나의 `ProtocolEntry`와 `__getitem__` 출력

crop/pad 판단은 리샘플 후 길이 기준이므로 native 초 단위와 target_sr 기준 샘플 수를 같이 계산한다.

## 셀 1 - Google Drive mount
Colab이 아니면 건너뛴다.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('[info] not on Colab - 로컬 경로로 진행합니다.')

## 셀 2 - 경로 / 환경 변수
Colab에서는 `DRIVE_ROOT`만 맞추면 된다. 로컬에서는 `data/dataset.py`가 있는 repo 루트를 위로 올라가며 찾는다.

In [ ]:
import os, sys

# 사용자 설정
DRIVE_ROOT = '/content/drive/MyDrive/STUDY/Project/ASVspoof5'

if IN_COLAB:
    PROJECT_ROOT = DRIVE_ROOT                   # 코드 위치 (Drive, 영구)
    DATA_ROOT    = '/content/data/asvspoof5'    # 추출된 flac/protocol (휘발)
    RAW_DIR      = '/content/raw'               # 다운받은 tar (휘발, 추출 즉시 삭제)
    HF_HOME      = f'{DRIVE_ROOT}/.hf_cache'
    os.environ['HF_HOME'] = HF_HOME
    os.environ['TRANSFORMERS_CACHE'] = HF_HOME
    os.environ['HF_HUB_CACHE'] = HF_HOME
else:
    # 로컬: 이 노트북이 있는 repo 루트를 자동 탐색
    def _find_repo_root(start=None):
        p = os.path.abspath(start or os.getcwd())
        while True:
            if os.path.isfile(os.path.join(p, 'data', 'dataset.py')):
                return p
            parent = os.path.dirname(p)
            if parent == p:
                raise RuntimeError('repo root(data/dataset.py)를 찾지 못했습니다.')
            p = parent
    PROJECT_ROOT = _find_repo_root()
    DATA_ROOT    = os.path.join(PROJECT_ROOT, 'data', 'asvspoof5')
    RAW_DIR      = os.path.join(PROJECT_ROOT, 'raw')

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('IN_COLAB     :', IN_COLAB)
print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_ROOT    :', DATA_ROOT, '(exists:', os.path.isdir(DATA_ROOT), ')')
print('RAW_DIR      :', RAW_DIR)

## 셀 3 - 데이터 다운로드 (Colab, flac_T + flac_D + protocols)
HF `jungjee/asvspoof5`에서 tar를 하나씩 받아 `DATA_ROOT`에 풀고 바로 지운다. 이미 있으면 skip. eval(`flac_E`)은 받지 않는다.

In [ ]:
import os, subprocess, shutil
from huggingface_hub import HfApi, hf_hub_download

REPO = "jungjee/asvspoof5"

def disk_free_gb(p="/content"):
    t, u, f = shutil.disk_usage(p); return f / 1024**3

def find_dir(name, root):
    for r, dirs, _ in os.walk(root):
        if name in dirs:
            return os.path.join(r, name)
    return None

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

if find_dir("flac_T", DATA_ROOT) and find_dir("flac_D", DATA_ROOT):
    print("[data] flac 이미 추출됨 → skip")
else:
    api = HfApi()
    files = api.list_repo_files(REPO, repo_type="dataset")
    protocols = sorted(f for f in files if f.startswith("ASVspoof5_protocols.tar"))
    train_sh  = sorted(f for f in files if f.startswith("flac_T_") and f.endswith(".tar"))
    dev_sh    = sorted(f for f in files if f.startswith("flac_D_") and f.endswith(".tar"))
    print(f"[stream] protocols={len(protocols)}  flac_T={len(train_sh)}  flac_D={len(dev_sh)}")

    def stream(fname):
        print(f"  {fname:<28} (free {disk_free_gb():5.0f}GB)", flush=True)
        path = hf_hub_download(REPO, fname, repo_type="dataset", local_dir=RAW_DIR)
        flag = "-xzf" if path.endswith(".gz") else "-xf"
        subprocess.run(["tar", flag, path, "-C", DATA_ROOT], check=True)
        os.remove(path)                          # 추출 즉시 tar 삭제 -> 디스크 절약
        shutil.rmtree(os.path.join(RAW_DIR, ".cache"), ignore_errors=True)  # hf 임시 blob 정리
        print(f"    extracted + tar 삭제, free {disk_free_gb():.0f}GB", flush=True)

    stream(protocols[0])                          # .tar.gz 우선 (정렬상 .gz가 뒤이면 조정)
    for f in train_sh: stream(f)
    for f in dev_sh:   stream(f)
    print("[stream] done")

## 셀 4 - config 로드, 경로 탐지
`configs/baseline.yaml`에서 `sample_rate`, `clip_seconds`를 읽어 기준 길이 `NUM_SAMPLES`를 계산한다. 추출 레이아웃이 다를 수 있어서 protocol과 flac 디렉터리는 `os.walk`로 찾는다.

In [ ]:
import yaml

with open(f'{PROJECT_ROOT}/configs/baseline.yaml') as f:
    cfg = yaml.safe_load(f)

TARGET_SR    = cfg['data']['sample_rate']
CLIP_SECONDS = cfg['data']['clip_seconds']
NUM_SAMPLES  = int(round(TARGET_SR * CLIP_SECONDS))   # 기준 길이 (crop/pad 판단 기준)
AUDIO_EXT    = cfg['data'].get('audio_ext', '.flac')

def find_file(name, root):
    for r, _, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    return None

def find_dir(prefix, root):
    for r, dirs, _ in os.walk(root):
        for d in dirs:
            if d.startswith(prefix):
                return os.path.join(r, d)
    return None

# split 이름 -> (protocol tsv 파일명, flac 디렉터리 prefix)
SPLIT_LOOKUP = {
    'train': ('ASVspoof5.train.tsv',        'flac_T'),
    'dev':   ('ASVspoof5.dev.track_1.tsv',  'flac_D'),
    'eval':  ('ASVspoof5.eval.track_1.tsv', 'flac_E'),
}

print('target_sr    :', TARGET_SR)
print('clip_seconds :', CLIP_SECONDS)
print('NUM_SAMPLES  :', NUM_SAMPLES, f'(= 기준 길이, {CLIP_SECONDS}s @ {TARGET_SR}Hz)')
print('AUDIO_EXT    :', AUDIO_EXT)

## 셀 5 - 프로토콜 로드
`parse_asvspoof5_protocol`로 split별 `ProtocolEntry` 리스트를 만든다. 오디오가 없어도 프로토콜만 있으면 로드된다.

In [ ]:
from data.protocols import parse_asvspoof5_protocol, ASVSPOOF5_COLUMNS

# 분석할 split 선택 (오디오가 큰 경우 'train'만으로 시작해도 됨)
SPLITS_TO_ANALYZE = ['train', 'dev']

splits = {}   # name -> {'entries', 'audio_dir'}
for name in SPLITS_TO_ANALYZE:
    tsv_name, flac_prefix = SPLIT_LOOKUP[name]
    proto_path = find_file(tsv_name, DATA_ROOT)
    audio_dir  = find_dir(flac_prefix, DATA_ROOT)
    if proto_path is None:
        print(f'[skip] {name}: 프로토콜({tsv_name}) 못 찾음 (DATA_ROOT 하위 검색)')
        continue
    entries = parse_asvspoof5_protocol(proto_path)
    splits[name] = {'entries': entries, 'audio_dir': audio_dir}
    print(f'[{name}] entries={len(entries):>7,}  proto={proto_path}')
    print(f'        audio_dir={audio_dir}  (exists={audio_dir is not None and os.path.isdir(audio_dir)})')

## 셀 6 - 라벨 / 성별 / 공격 분포 (프로토콜만 사용)

In [ ]:
from collections import Counter

for name, s in splits.items():
    ent = s['entries']
    print(f'=== {name}  (n={len(ent):,}) ===')
    print('  key      :', dict(Counter(e.key for e in ent)))
    print('  label    :', dict(Counter(e.label for e in ent)), '  (0=bonafide, 1=spoof, -1=hidden)')
    print('  gender   :', dict(Counter(e.speaker_gender for e in ent)))
    print('  codec    :', dict(Counter(e.codec for e in ent)))
    top_atk = Counter(e.attack_label for e in ent).most_common(10)
    print('  attack   :', top_atk, '... (top 10)')
    print()

## 셀 7 - 오디오 메타데이터 (샘플레이트, 길이)
`soundfile.info()`는 헤더만 읽어서 빠르다. 파일별 `samplerate`, `frames`를 모은 뒤 리샘플을 반영한 target_sr 기준 샘플 수(`frames_resampled`)를 계산한다.
`LIMIT`을 정수로 두면 앞에서부터 그만큼만 본다.

In [ ]:
import time
import numpy as np
import pandas as pd
import soundfile as sf

LIMIT = None          # 예: 5000 (빠른 미리보기). None = 전체.
PRINT_EVERY = 20000

def collect_meta(name, entries, audio_dir, limit=None):
    rows, missing = [], 0
    subset = entries[:limit] if limit else entries
    t0 = time.time()
    for i, e in enumerate(subset):
        path = os.path.join(audio_dir, f'{e.file_id}{AUDIO_EXT}')
        try:
            info = sf.info(path)
        except (RuntimeError, FileNotFoundError):
            missing += 1
            continue
        rows.append((e.file_id, e.key, e.label, info.samplerate, info.frames))
        if PRINT_EVERY and (i + 1) % PRINT_EVERY == 0:
            print(f'  [{name}] {i+1:,}/{len(subset):,}  ({time.time()-t0:.0f}s)')
    df = pd.DataFrame(rows, columns=['file_id', 'key', 'label', 'samplerate', 'frames'])
    # native 초 단위
    df['seconds'] = df['frames'] / df['samplerate']
    # _load_audio 리샘플 반영: target_sr 기준 유효 샘플 수
    #   sr == TARGET_SR 이면 frames 그대로; 아니면 round(frames * TARGET_SR/sr)
    df['frames_resampled'] = np.where(
        df['samplerate'] == TARGET_SR,
        df['frames'],
        np.round(df['frames'] * (TARGET_SR / df['samplerate'])).astype('int64'),
    )
    print(f'[{name}] collected={len(df):,}  missing_audio={missing:,}  ({time.time()-t0:.0f}s)')
    return df

meta = {}
for name, s in splits.items():
    if not (s['audio_dir'] and os.path.isdir(s['audio_dir'])):
        print(f'[skip] {name}: 오디오 폴더 없음 -> {s["audio_dir"]}')
        continue
    meta[name] = collect_meta(name, s['entries'], s['audio_dir'], LIMIT)

## 분석 1 - 샘플레이트 분포
target_sr 하나만 있으면 선형보간 리샘플 경로는 타지 않는다.

In [ ]:
for name, df in meta.items():
    vc = df['samplerate'].value_counts().sort_index()
    n_off = int((df['samplerate'] != TARGET_SR).sum())
    print(f'=== {name} 샘플레이트 분포 (target_sr={TARGET_SR}) ===')
    for sr, c in vc.items():
        flag = '' if sr == TARGET_SR else '  <-- 리샘플 발생(선형보간)'
        print(f'  {sr:>7} Hz : {c:>8,}{flag}')
    print(f'  target_sr 아닌 파일 수: {n_off:,} / {len(df):,}\n')

## 분석 2 - 길이 분포, 기준 길이 대비 미달/초과
- `frames_resampled >= NUM_SAMPLES`: crop
- `frames_resampled < NUM_SAMPLES`: pad

In [ ]:
for name, df in meta.items():
    fr = df['frames_resampled']
    sec = fr / TARGET_SR
    n = len(df)
    n_crop = int((fr >= NUM_SAMPLES).sum())
    n_pad  = int((fr <  NUM_SAMPLES).sum())
    print(f'=== {name}  (n={n:,}, NUM_SAMPLES={NUM_SAMPLES} = {CLIP_SECONDS}s) ===')
    print(f'  길이(초, target_sr 기준)  min={sec.min():.3f}  median={sec.median():.3f}  '
          f'mean={sec.mean():.3f}  max={sec.max():.3f}')
    print(f'  분위수(초)  p1={sec.quantile(.01):.3f}  p5={sec.quantile(.05):.3f}  '
          f'p50={sec.quantile(.50):.3f}  p95={sec.quantile(.95):.3f}  p99={sec.quantile(.99):.3f}')
    print(f'  >= 기준(크롭) : {n_crop:>8,}  ({n_crop/n*100:.2f}%)')
    print(f'  <  기준(패딩) : {n_pad:>8,}  ({n_pad/n*100:.2f}%)')
    print()

In [ ]:
# 길이 히스토그램 (초 단위, 기준 길이 표시)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(meta), figsize=(6 * max(len(meta), 1), 4), squeeze=False)
for ax, (name, df) in zip(axes[0], meta.items()):
    sec = df['frames_resampled'] / TARGET_SR
    ax.hist(sec, bins=100)
    ax.axvline(CLIP_SECONDS, color='r', ls='--', label=f'기준 {CLIP_SECONDS}s')
    ax.set_title(f'{name}  길이 분포 (초)')
    ax.set_xlabel('seconds'); ax.set_ylabel('count'); ax.legend()
plt.tight_layout(); plt.show()

## 분석 3 - 패딩 대상 중 reflect / constant
패딩은 기준 길이 미달 파일에만 적용된다.
- `frames_resampled >= 2`: reflect pad
- `frames_resampled < 2`: constant(0) pad, 사실상 무음이라 있으면 확인 필요

In [ ]:
for name, df in meta.items():
    fr = df['frames_resampled']
    pad_mask = fr < NUM_SAMPLES
    n_pad = int(pad_mask.sum())
    n_reflect  = int((pad_mask & (fr >= 2)).sum())
    n_constant = int((pad_mask & (fr <  2)).sum())
    print(f'=== {name}  패딩 분해 (총 패딩 대상 {n_pad:,}) ===')
    print(f'  reflect(대칭) 패딩  (len>=2) : {n_reflect:>8,}')
    print(f'  constant(0)  패딩  (len<2)  : {n_constant:>8,}'
          + ('   <-- 무음 부착 발생! 점검 필요' if n_constant else '   (없음, OK)'))
    if n_constant:
        bad = df[pad_mask & (fr < 2)][['file_id', 'frames', 'samplerate', 'frames_resampled']]
        print('  해당 파일 예시:')
        print(bad.head(20).to_string(index=False))
    print()

## 분석 4 - 샘플 하나 확인
`ASVspoof5Track1Dataset`으로 한 샘플을 로드해서 `ProtocolEntry` 필드와 `__getitem__` 반환값을 본다.

In [ ]:
from dataclasses import asdict
from data.dataset import ASVspoof5Track1Dataset, SplitInfo

INSPECT_SPLIT = 'train' if 'train' in splits else next(iter(splits))
INSPECT_IDX   = 0

s = splits[INSPECT_SPLIT]
entry = s['entries'][INSPECT_IDX]
print('=== ProtocolEntry 필드 ===')
for k, v in asdict(entry).items():
    print(f'  {k:15s}: {v!r}')

In [ ]:
# is_train=False(중앙 크롭, augment 없음)로 결정론적 로드
ds = ASVspoof5Track1Dataset(
    SplitInfo(entries=s['entries'], audio_dir=s['audio_dir'], name=INSPECT_SPLIT),
    sample_rate=TARGET_SR,
    clip_seconds=CLIP_SECONDS,
    audio_ext=AUDIO_EXT,
    augment=None,
    is_train=False,
)
print('len(ds) :', len(ds))
sample = ds[INSPECT_IDX]

print('\n=== __getitem__ 반환 ===')
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(f'  {k:10s}: shape={tuple(v.shape)}  dtype={v.dtype}  '
              f'min={v.min().item():.4f} max={v.max().item():.4f}')
    else:
        print(f'  {k:10s}: {v!r}')
assert sample['distorted'].shape[0] == NUM_SAMPLES, '크롭/패딩 후 길이가 기준과 다름!'

In [ ]:
# 파형 시각화
import matplotlib.pyplot as plt

wav = sample['clean'].numpy()
t = np.arange(len(wav)) / TARGET_SR
plt.figure(figsize=(12, 3))
plt.plot(t, wav, lw=0.5)
plt.title(f"{INSPECT_SPLIT}[{INSPECT_IDX}]  file_id={sample['file_id']}  key={sample['key']}  label={sample['label'].item()}")
plt.xlabel('seconds'); plt.ylabel('amplitude'); plt.tight_layout(); plt.show()